This notebook covers dataset preparation for the fake news classification task.

Steps:
- Load and merge Kaggle datasets (real + fake news)
- Assign binary labels (1 = real, 0 = fake)
- Remove short articles
- Remove duplicates
- Clean `(Reuters)` tag to avoid [source](https://www.reuters.com/) leakage

Final dataset is saved as a pickle file for further analysis and modeling.

In [7]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent.resolve()))

In [8]:
import pandas as pd
from dotenv import load_dotenv
from kaggle.api.kaggle_api_extended import KaggleApi

from src.env import DATA_DIR, DATASET_PATH

In [9]:
load_dotenv()
api = KaggleApi()
api.authenticate()

api.dataset_download_files(
    "clmentbisaillon/fake-and-real-news-dataset", path=DATA_DIR, unzip=True
)

Dataset URL: https://www.kaggle.com/datasets/clmentbisaillon/fake-and-real-news-dataset


In [10]:
list(DATA_DIR.iterdir())

[WindowsPath('D:/Programming/projects/recruitment-task/data/Fake.csv'),
 WindowsPath('D:/Programming/projects/recruitment-task/data/True.csv')]

In [11]:
fake_df = pd.read_csv(DATA_DIR / "Fake.csv")
true_df = pd.read_csv(DATA_DIR / "True.csv")

print(f"Fake columns: {fake_df.columns.tolist()}. {len(fake_df)} rows.")
print(f"True columns: {true_df.columns.tolist()}. {len(true_df)} rows.")

Fake columns: ['title', 'text', 'subject', 'date']. 23481 rows.
True columns: ['title', 'text', 'subject', 'date']. 21417 rows.


In [12]:
fake_df["label"] = 0
true_df["label"] = 1

df = pd.concat(
    [
        fake_df,
        true_df,
    ],
    ignore_index=True,
)

In [13]:
df["text_length"] = df["text"].astype(str).apply(len)
df["word_count"] = df["text"].astype(str).apply(lambda x: len(x.split()))
df = df[df["word_count"] >= 100]

In [14]:
df.duplicated(subset=["text"]).sum()

np.int64(4975)

In [15]:
df = df.drop_duplicates(subset=["text"])

In [16]:
df

,title,text,subject,date,label,text_length,word_count
0,Donald Trump Sends Out Embarrassing New Year’...,Donald Trump just couldn t wish all Americans ...,News,"December 31, 2017",0,2893,495
1,Drunk Bragging Trump Staffer Started Russian ...,House Intelligence Committee Chairman Devin Nu...,News,"December 31, 2017",0,1898,305
2,Sheriff David Clarke Becomes An Internet Joke...,"On Friday, it was revealed that former Milwauk...",News,"December 30, 2017",0,3597,580
3,Trump Is So Obsessed He Even Has Obama’s Name...,"On Christmas day, Donald Trump announced that ...",News,"December 29, 2017",0,2774,444
4,Pope Francis Just Called Out Donald Trump Dur...,Pope Francis used his annual Christmas Day mes...,News,"December 25, 2017",0,2346,420
...,...,...,...,...,...,...,...
44893,'Fully committed' NATO backs new U.S. approach...,BRUSSELS (Reuters) - NATO allies on Tuesday we...,worldnews,"August 22, 2017",1,2821,466
44894,LexisNexis withdrew two products from Chinese ...,"LONDON (Reuters) - LexisNexis, a provider of l...",worldnews,"August 22, 2017",1,800,125
44895,Minsk cultural hub becomes haven from authorities,MINSK (Reuters) - In the shadow of disused Sov...,worldnews,"August 22, 2017",1,1950,320
44896,Vatican upbeat on possibility of Pope Francis ...,MOSCOW (Reuters) - Vatican Secretary of State ...,worldnews,"August 22, 2017",1,1199,205


In [17]:
keyword = "reuters"

contains_reuters = df["text"].str.contains(keyword, case=False, na=False)

df["contains_reuters"] = contains_reuters

reuters_counts = (
    df.groupby("label")["contains_reuters"].sum().rename({0: "Fake", 1: "Real"})
)

print("Articles containing 'Reuters':")
print(reuters_counts)

class_sizes = df["label"].value_counts().sort_index()

reuters_percentages = reuters_counts / class_sizes.values * 100

print("\nPercentage of articles containing 'Reuters':")
print(reuters_percentages.round(2))

Articles containing 'Reuters':
label
Fake      218
Real    17593
Name: contains_reuters, dtype: int64

Percentage of articles containing 'Reuters':
label
Fake     1.35
Real    99.79
Name: contains_reuters, dtype: float64


In [18]:
real_examples = df[(df["label"] == 1) & (df["contains_reuters"])]["text"].sample(3)

print("REAL examples containing Reuters:\n")

for i, text in enumerate(real_examples, 1):
    print(f"\n--- Example {i} ---")
    print(text[:1000])

REAL examples containing Reuters:


--- Example 1 ---
PARIS (Reuters) - Fewer French voters are dissatisfied with Emmanuel Macron s performance, a poll showed on Sunday, halting a recent slide in the popularity ratings of the French president in recent months. The poll, conducted by Ifop for newspaper Le Journal du Dimanche (JDD), showed Macron s  dissatisfaction rating  declining to 53 percent in September, from 57 percent in August. Some 45 percent expressed satisfaction with the centrist leader - up from 40 percent in August. The poll of 1,989 people was carried out on Sept. 15-23. Macron s approval ratings have dropped sharply in opinion polls since his election in May, dragged down by labor reforms and planned budget cuts, including a decrease in housing aid for students. The new poll comes as French far-left opposition party leader Jean-Luc Melenchon drew tens of thousands to a rally on Saturday against Macron s labor reforms, aiming to reinforce his credentials as Macron s stron

In [19]:
reuters_prefix_pattern = r"^[A-Z\s]+ \(Reuters\) -"

matches_reuters_prefix = df["text"].str.contains(
    reuters_prefix_pattern, regex=True, na=False
)

count_matches = matches_reuters_prefix.sum()
percentage_matches = matches_reuters_prefix.mean() * 100

print(f"Rows matching Reuters prefix pattern: {count_matches}")
print(f"Percentage of dataset: {percentage_matches:.2f}%")

Rows matching Reuters prefix pattern: 13770
Percentage of dataset: 40.82%


In [20]:
df["text"] = df["text"].str.replace(reuters_prefix_pattern, "", regex=True).str.strip()

df["text"] = df["text"].str.replace("(Reuters)", "", regex=True).str.strip()

In [21]:
keyword = "reuters"

contains_reuters = df["text"].str.contains(keyword, case=False, na=False)

df["contains_reuters"] = contains_reuters

reuters_counts = (
    df.groupby("label")["contains_reuters"].sum().rename({0: "Fake", 1: "Real"})
)

print("Articles containing 'Reuters':")
print(reuters_counts)

class_sizes = df["label"].value_counts().sort_index()

reuters_percentages = reuters_counts / class_sizes.values * 100

print("\nPercentage of articles containing 'Reuters':")
print(reuters_percentages.round(2))

Articles containing 'Reuters':
label
Fake     7
Real    44
Name: contains_reuters, dtype: int64

Percentage of articles containing 'Reuters':
label
Fake    0.04
Real    0.25
Name: contains_reuters, dtype: float64


In [23]:
df

,title,text,subject,date,label,text_length,word_count,contains_reuters
0,Donald Trump Sends Out Embarrassing New Year’...,Donald Trump just couldn t wish all Americans ...,News,"December 31, 2017",0,2893,495,False
1,Drunk Bragging Trump Staffer Started Russian ...,House Intelligence Committee Chairman Devin Nu...,News,"December 31, 2017",0,1898,305,False
2,Sheriff David Clarke Becomes An Internet Joke...,"On Friday, it was revealed that former Milwauk...",News,"December 30, 2017",0,3597,580,False
3,Trump Is So Obsessed He Even Has Obama’s Name...,"On Christmas day, Donald Trump announced that ...",News,"December 29, 2017",0,2774,444,False
4,Pope Francis Just Called Out Donald Trump Dur...,Pope Francis used his annual Christmas Day mes...,News,"December 25, 2017",0,2346,420,False
...,...,...,...,...,...,...,...,...
44893,'Fully committed' NATO backs new U.S. approach...,NATO allies on Tuesday welcomed President Dona...,worldnews,"August 22, 2017",1,2821,466,False
44894,LexisNexis withdrew two products from Chinese ...,"LexisNexis, a provider of legal, regulatory an...",worldnews,"August 22, 2017",1,800,125,False
44895,Minsk cultural hub becomes haven from authorities,In the shadow of disused Soviet-era factories ...,worldnews,"August 22, 2017",1,1950,320,False
44896,Vatican upbeat on possibility of Pope Francis ...,Vatican Secretary of State Cardinal Pietro Par...,worldnews,"August 22, 2017",1,1199,205,False


In [ ]:
df["text"] = df["text"].astype("object")
df[["text", "label", "text_length", "word_count"]].to_parquet(DATASET_PATH, engine="fastparquet", index=False)
print("Saved to:", DATASET_PATH)

Saved to: D:\Programming\projects\recruitment-task\data\dataset.parquet


In [31]:
for file_path in DATA_DIR.glob("*.csv"):
    file_path.unlink()
    print(f"Deleted: {file_path.name}")

Deleted: Fake.csv
Deleted: True.csv
